# Random Forest
Εκπαιδεύουμε πολλά variations του μοντέλου με διαφορετικές υπερπαραμέτρους και βρίσκουμε το μοντέλο με τις καλύτερες υπερπαραμέτρους. Ύστερα, το εφαρμόζουμε πάνω στο test gold για να κάνουμε προβλέψεις και τις αποθηκεύουμε για να γίνουν evaluate μαζί με όλες τις άλλες 

In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator

print("1. Εκκίνηση SparkSession...")
spark = SparkSession.builder \
    .appName("RF_SMOTE_Corrected_GridSearch") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

print("2. Φόρτωση Δεδομένων (Απευθείας από το ΔΙΟΡΘΩΜΕΝΟ SMOTE Gold dataset)...")
train_gold = spark.read.parquet("../../data/train_gold_corrected.parquet")
test_gold = spark.read.parquet("../../data/test_gold_corrected.parquet")

train_gold = train_gold.withColumn("stroke", train_gold["stroke"].cast(DoubleType()))
test_gold = test_gold.withColumn("stroke", test_gold["stroke"].cast(DoubleType()))
train_gold = train_gold.withColumn("cluster", F.col("cluster").cast(DoubleType()))
test_gold = test_gold.withColumn("cluster", F.col("cluster").cast(DoubleType()))

assembler = VectorAssembler(inputCols=["features", "cluster"], outputCol="augmented_features")
train_augmented = assembler.transform(train_gold).drop("features").withColumnRenamed("augmented_features", "features")
test_augmented = assembler.transform(test_gold).drop("features").withColumnRenamed("augmented_features", "features")

# Κρατάμε ολόκληρο το SMOTE dataset (χωρίς undersampling)
train_augmented.cache()

print("3. Ορισμός Πλέγματος Παραμέτρων (Grid Search)...")
rf_base = RandomForestClassifier(featuresCol="features", labelCol="stroke", seed=22390225)

# ΟΙ ΑΛΛΑΓΕΣ:
# - maxDepth [3, 5, 7]: Πιο ρηχά δέντρα για αποφυγή αποστήθισης του SMOTE.
# - numTrees [50, 100, 150]: Τα 200 δέντρα συχνά ομαλοποιούν υπερβολικά τις προβλέψεις υπέρ της πλειοψηφικής κλάσης.
# - featureSubsetStrategy: Αναγκάζει τα δέντρα να επιλέγουν τυχαία υποσύνολα χαρακτηριστικών, αυξάνοντας το Recall της Class 1.
paramGrid = (ParamGridBuilder()
             .addGrid(rf_base.maxDepth, [3, 5, 7]) 
             .addGrid(rf_base.numTrees, [50, 100, 150])
             .addGrid(rf_base.featureSubsetStrategy, ["sqrt", "log2", "0.2"])
             .build())

# ΔΙΟΡΘΩΣΗ: Αλλαγή σε areaUnderROC για να μεγιστοποιήσουμε το Recall και τη συνολική διαχωριστική ικανότητα
evaluator = BinaryClassificationEvaluator(
    labelCol="stroke", 
    rawPredictionCol="rawPrediction", 
    metricName="areaUnderROC"
)

cv = CrossValidator(estimator=rf_base,
                    estimatorParamMaps=paramGrid,
                    evaluator=evaluator,
                    numFolds=3,
                    seed=22390225)

print("4. Εκτέλεση Cross-Validation στο ΠΛΗΡΕΣ SMOTE Dataset...")
cv_model = cv.fit(train_augmented)
best_rf = cv_model.bestModel

print("\n===========================================================")
print("[ΔΙΟΡΘΩΜΕΝΗ ΕΥΡΕΣΗ SMOTE] Οι βέλτιστες παράμετροι βρέθηκαν:")
print(f" -> maxDepth: {best_rf._java_obj.getMaxDepth()}")
print(f" -> numTrees: {best_rf._java_obj.getNumTrees()}")
print("===========================================================")

print("5. Παραγωγή και αποθήκευση προβλέψεων στο Test Set...")
rf_preds = best_rf.transform(test_augmented)

# Αποθήκευση ως το κεντρικό rf_predictions
rf_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../../data/rf_predictions.parquet")

spark.stop()
print("Ολοκληρώθηκε! Έχεις πλέον ένα RF που αντιστέκεται στο overfitting του SMOTE.")

1. Εκκίνηση SparkSession...
2. Φόρτωση Δεδομένων (Απευθείας από το ΔΙΟΡΘΩΜΕΝΟ SMOTE Gold dataset)...
3. Ορισμός Πλέγματος Παραμέτρων (Grid Search)...
4. Εκτέλεση Cross-Validation στο ΠΛΗΡΕΣ SMOTE Dataset...


26/06/12 03:44:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/12 03:44:26 WARN DAGScheduler: Broadcasting large task binary with size 1421.8 KiB
26/06/12 03:44:26 WARN DAGScheduler: Broadcasting large task binary with size 1173.1 KiB
26/06/12 03:44:27 WARN DAGScheduler: Broadcasting large task binary with size 1421.8 KiB
26/06/12 03:44:27 WARN DAGScheduler: Broadcasting large task binary with size 1173.1 KiB
26/06/12 03:44:28 WARN DAGScheduler: Broadcasting large task binary with size 1357.7 KiB
26/06/12 03:44:28 WARN DAGScheduler: Broadcasting large task binary with size 1105.0 KiB
26/06/12 03:44:29 WARN DAGScheduler: Broadcasting large task binary with size 1249.7 KiB
26/06/12 03:44:29 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
26/06/12 03:44:30 WARN DAGScheduler: Broadcasting large task binary with size 1716.7 KiB
26/06/12 03:44:30 WARN DAGScheduler: Broadcasting large task binary with size 1249.7 KiB
26/06/12 03:44


[ΔΙΟΡΘΩΜΕΝΗ ΕΥΡΕΣΗ SMOTE] Οι βέλτιστες παράμετροι βρέθηκαν:
 -> maxDepth: 7
 -> numTrees: 50
5. Παραγωγή και αποθήκευση προβλέψεων στο Test Set...
Ολοκληρώθηκε! Έχεις πλέον ένα RF που αντιστέκεται στο overfitting του SMOTE.
